In [ ]:
import torch.nn as nn
from src.utils.models import *

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
class Scheduler(nn.Module):
    def __init__(self, num_epochs, beta_init=1e-4, beta_finish=0.02):
        super().__init__()
        self.beta = torch.linspace(beta_init, beta_finish, num_epochs)
        alpha = 1 - self.beta
        self.alpha = torch.cumprod(alpha, dim=0).requires_grad_(False)
        
    def forward(self, t):
        return self.beta[t], self.alpha[t]

In [ ]:
class Diffuser(nn.Module):
    def __init__(self, model, scheduler):
        super().__init__()
        self.model = model
        self.scheduler = scheduler

    # x: input image, t: time step
    def forward(self, x, t):
        # e = torch.randn(1, 1, 32, 32) # ruido gaussiano
        e = torch.randn_like(x)  # ruido gaussiano mismo tamaño que x
        beta_t, alpha_t = self.scheduler(t)
        # para que vaya con tensores
        beta_t = beta_t.view(-1, 1, 1, 1)
        alpha_t = alpha_t.view(-1, 1, 1, 1)
        z = torch.sqrt(alpha_t) * x + torch.sqrt(beta_t) * e
        return z, e    

In [ ]:
class Embeder(nn.Module):
    def __init__(self, num_epochs, embed_dim):
        super().__init__()
        position = torch.arange(num_epochs).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, embed_dim, 2).float() * -(math.log(10000.0) / embed_dim))
        embeddings = torch.zeros(num_epochs, embed_dim, requires_grad=False)
        embeddings[:, 0::2] = torch.sin(position * div)
        embeddings[:, 1::2] = torch.cos(position * div)
        self.embeddings = embeddings

    def forward(self, x, t):
        embeds = self.embeddings[t].to(x.device)
        return embeds[:, :, None, None]

In [ ]:
'''
    La vaina es la siguiente:
    - input_channels: canales de entrada (1 para escala de grises, 3 para RGB)
    - output_channels: canales de salida (1 para escala de grises, 3 para RGB)
    - channels: ni idea
    - layers: capas que van a hacer vainas a la entrada, la clase aun no esta implementada -> la entrada coincide con channels 0
        -> la salida coincide con channels 1
    - embedder: objeto que va a hacer el embedding del tiempo
    - channels: 
        0. canales de salida de la primera convolucion = canales de entrada a las layers
        1. canales de salida de las layers = canales de entrada de la convolucion de salida
'''
class DiffusionModel(nn.Module):
    def __init__(self, channels:list, layers:list[nn.Module], embedder:nn.Module, input_channels=1, output_channels=1):
        super().__init__()
        self.relu = nn.ReLU()
        self.channels = channels
        
        # aparentemente esto va de hacer redes convolucionales
        self.conv_in = nn.Conv2d(input_channels, channels[0], kernel_size=3, padding=1)
        
        self.conv_out = nn.Conv2d(channels[1], output_channels, kernel_size=3, padding=1)
        
        self.embeder = embedder
        self.layers = layers
        
    def forward(self, x, t):
        B, C, H, W = x.shape # batch, canales, altura, anchura
        # 1. pasar por la convolucion 1
        h = self.relu(self.conv_in(x))
        # 2. obtener el embedding del tiempo
        t_emb = self.embeder(x, t)
        # t_emb = t_emb.view(B, -1, 1, 1)
        # 3. pasar por las layers
        for layer in self.layers:
            h = layer(h, t_emb)
        # 4. pasar por la convolucion de salida
        out = self.conv_out(h)
        return out
    
    
         